In [ ]:
import numpy as np
import andes
from andes.utils.paths import get_case

import matplotlib.pyplot as plt
# For Jupyter notebook
%matplotlib inline

In [ ]:
andes.config_logger(stream_level=40) # 10-DEBUG, 20-INFO, 30-WARNING, 40-ERROR, 50-CRITICAL

In [ ]:
ss = andes.load(get_case('wecc/wecc_full.xlsx'), setup=False)

In [ ]:
ss.GENROU.as_df()

In [ ]:
ss.Bus.as_df()

In [ ]:
ss.Fault.as_df()

In [ ]:
ss.IEEEG1.as_df()

The IEEEG1 model currently does not support modifying the reference value! https://docs.andes.app/en/latest/groupdoc/TurbineGov.html#ieeeg1

IEEG1 model references:

- https://www.powerworld.com/WebHelp/Content/TransientModels_HTML/Governor%20IEEEG1,%20IEEEG1D%20and%20IEEEG1_GE.htm
- https://ww2.mathworks.cn/help/sps/ref/governortype1.html

In [ ]:
# idxes = [4,5,7,9] # buses connected to induction (non-synchronous) generators
ss = andes.load(get_case('wecc/wecc_full.xlsx'), setup=False)
# ss.Fault.alter('bus', 1, idx)  # arguments are `src`, `idx`, `value
# ss.Fault.alter('tc',  1, 1.2)  # arguments are `src`, `idx`, `value
# ss.add("Toggle", dict(model='SynGen', dev=ss.GENROU.idx.v[idx], t=1.0))
# ss.add("Toggle", dict(model='SynGen', dev=dict1['idx'], t=1.0))
ss.setup()
ss.PFlow.run()

ss.TDS.config.tf = 10
ss.TDS.run()
if ss.exit_code == 0:
    condition_initial = ss.IEEEG1.pref0.v

In [ ]:
condition_initial

In [ ]:
from typing import Any

results = dict[Any, Any]()
condition_record = dict[int, Any]()
ss = andes.load(get_case('wecc/wecc_full.xlsx'), setup=False)

ref0_amount = 0.2
fault_idx = 0
idx = 0
ss = andes.load(get_case('wecc/wecc_full.xlsx'), setup=False)
ss.add('Alter', param_dict = {'name':ss.IEEEG1.name.v[idx],  't':1.0, 'model':'IEEEG1', 'dev':ss.IEEEG1.idx.v[idx], 'src': 'pref0', 'method' :'*', 'amount':ref0_amount})
ss.setup()
ss.PFlow.run()

ss.TDS.config.tf = 10
ss.TDS.run()
if ss.exit_code == 0:
    results[fault_idx*len(ss.IEEEG1) + idx] = ss
    condition_record[fault_idx*len(ss.IEEEG1) + idx] = ss.IEEEG1.pref0.v

In [ ]:
ss.Alter.as_df()

In [ ]:
fault_idx = list(results.keys())[0]
results[fault_idx].TDS.plt.plot(results[fault_idx].TDS.plt.find('omega')[0])

Load increase disturbance

In [ ]:
from typing import Any


import os

ss = andes.load(get_case('wecc/wecc_full.xlsx'), setup=False)

results = dict[Any, Any]()
condition_record = dict[int, Any]()

for idx in range(len(ss.PQ)):  
    ss = andes.load(get_case('wecc/wecc_full.xlsx'), setup=False)
    ss.add('Alter', param_dict = {'name':ss.PQ.name.v[idx],  't':1.0, 'model':'PQ', 'dev':ss.PQ.idx.v[idx], 
        'src': 'Req', 'method' :'*', 'amount':2.0})
    ss.add('Alter', param_dict = {'name':ss.PQ.name.v[idx],  't':1.0, 'model':'PQ', 'dev':ss.PQ.idx.v[idx], 
        'src': 'Xeq', 'method' :'*', 'amount':2.0})    # ss.add("Toggle", dict(model='SynGen', dev=ss.GENROU.idx.v[idx], t=1.0))
    # ss.add("Toggle", dict(model='SynGen', dev=dict1['idx'], t=1.0))
    ss.setup()
    ss.PFlow.run()

    ss.TDS.config.tf = 10
    ss.TDS.run()
    if ss.exit_code == 0:
        results[idx] = ss
        condition_record[idx] = np.concatenate((ss.PQ.Req.v, ss.PQ.Xeq.v))
    
for ii in results.keys():
    results[ii].TDS.plt.export_csv(os.path.join('wecc_multi_traj', 'wecc_increase_load_out_'+str(ii)+'.csv'))
print(results.keys())

In [ ]:
condition_record[0].shape

In [ ]:
import csv
import os
# Write condition_record to a CSV file
csv_filename = os.path.join('wecc_multi_traj','condition_record_wecc.csv')
with open(csv_filename, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    # Header: first column is key, followed by the elements of value
    writer.writerow(['key'] + [f'value{i}' for i in range(1, 104*2)])
    # Iterate over each item in condition_record
    for key, value_list in condition_record.items():
        # Pad with empty strings if needed
        row = [key] + list(value_list) + [''] * max(0, 5 - len(value_list))
        writer.writerow(row)

In [ ]:
fault_idx = list(results.keys())[0]
results[fault_idx].TDS.plt.plot(results[fault_idx].TDS.plt.find('omega')[0])

In [ ]:
fault_idx = list(results.keys())[0]
results[fault_idx].TDS.plt.plot(results[fault_idx].TDS.plt.find('delta')[0])

In [ ]:
fault_idx = list(results.keys())[8]
results[fault_idx].TDS.plt.plot(results[fault_idx].TDS.plt.find('omega')[0])

In [ ]:
fault_idx = list(results.keys())[8]
results[fault_idx].TDS.plt.plot(results[fault_idx].TDS.plt.find('delta')[0])